# Домашнее задание 1. Классификация сценариев: TF-IDF и эмбеддинги
Блок 1 «Основы NLP и векторные представления»
Голосовой помощник получает фразу и должен понять, какому навыку её отдать: будильнику, погоде, почте
или какому-то другому. Ошибка маршрутизации — и пользователь слышит прогноз погоды вместо звонка маме. Главный вопрос
домашки звучит на каждом проекте: **нужна ли здесь нейросеть, или хватит быстрого и дешёвого TF-IDF?**


На семинаре мы различали 8 интентов голосового помощника. В домашнем задании задача шире:
определить **сценарий** запроса (`scenario`) среди 18 классов MASSIVE ru-RU — будильник, календарь,
музыка, погода, почта, умный дом и т. д. Классы неравномерны и местами пересекаются по словам.

Сначала соберём классификатор на TF-IDF, потом на векторах предложений из двух готовых моделей
и сравним их. Затем посмотрим, сколько размеченных примеров нужно каждому подходу и на каких запросах
они ошибаются.

**Как сдавать.** Почти весь код уже написан. В каждом задании с кодом есть одна функция или ячейка
с 3–5 пропусками: на месте `...` нужно дописать код, что именно — сказано в комментарии `TODO`.
Выполните ноутбук сверху вниз (Run All)
и сдайте `.ipynb` с сохранёнными выводами. На вопросы хватит одного-двух предложений в ячейках «Ответ».
Не меняйте разбиение данных и `SEED`: иначе результаты нельзя сравнить.

| Задание | Баллы |
|---|---:|
| 1. Нормализация и стемминг | 1,5 |
| 2. TF-IDF: слова и символы, подбор `C` | 2 |
| 3. Векторы предложений двух моделей | 2 |
| 4. k ближайших соседей своими руками | 1 |
| 5. Оценка на test и confusion matrix | 1,5 |
| 6. Кривая обучения | 1 |
| 7. Анализ ошибок и выводы | 1 |
| **Итого** | **10** |

**Как оценивается каждое задание.**

* Пропуски в коде заполнены, ноутбук выполняется сверху вниз без ошибок, проверки `assert` проходят,
  выводы сохранены.
* Ответ на вопрос опирается на числа из вашего запуска.
* На проверке преподаватель задаёт вопросы по коду задания — и по дописанным строкам, и по готовому коду
  вокруг них: что он делает и зачем. Если вы не ориентируетесь в коде, баллы за задание снижаются.

In [ ]:
# requirements (Python 3.10–3.12); в скобках — версии, на которых ноутбук прогнан:
#   numpy>=1.26                  (проверено 2.5.4)
#   pandas>=2.2                  (проверено 3.1.1)
#   scikit-learn>=1.5            (проверено 1.9.1)
#   matplotlib>=3.8              (проверено 3.11.2)
#   nltk>=3.9                    (проверено 3.10.4)
#   requests>=2.31               (проверено 2.34.2)
#   torch>=2.4                   (проверено 2.15.0)
#   sentence-transformers>=3.0   (проверено 6.2.0)
# Установка локально: pip install -r requirements.txt
# Google Colab: раскомментируйте строку ниже и перезапустите среду после установки.
# %pip install -q "numpy>=1.26" "pandas>=2.2" "scikit-learn>=1.5" "matplotlib>=3.8" "nltk>=3.9" "requests>=2.31" "torch>=2.4" "sentence-transformers>=3.0"
#
# GPU не обязателен: кодирование ~10 тыс. коротких фраз на CPU занимает несколько минут.

import sys
print("Python", sys.version.split()[0])

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re, json, random, hashlib, tarfile, unicodedata
from pathlib import Path
from time import perf_counter
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from IPython.display import display
from nltk.tokenize import wordpunct_tokenize
from nltk.stem import SnowballStemmer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
CACHE_DIR = Path.home() / ".cache" / "nlp_course"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 100)
print("device:", DEVICE)

## 0. Данные (готовый код)
Повторы удаляем специально: если одна и та же фраза попадёт и в train, и в test, модель просто
«вспомнит» её, и метрика окажется завышенной.

Тот же архив MASSIVE 1.1, что на семинаре, с проверкой SHA-256. Метка — `scenario`.
Для скорости обучающая часть ограничена 300 примерами на класс; dev и test берутся целиком
после удаления повторов. Повторы удаляются **между** частями: одна и та же фраза не может
оказаться и в train, и в test.

In [ ]:
DATA_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
ARCHIVE_SHA256 = "4cba5faa11c71437928e17cb1b9b3d8b8e727e7ea363a3a9a8045e19c0491577"
ARCHIVE_PATH = CACHE_DIR / "amazon-massive-dataset-1.1.tar.gz"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

if not ARCHIVE_PATH.exists():
    tmp = ARCHIVE_PATH.with_suffix(".download")
    with requests.get(DATA_URL, stream=True, timeout=(20, 300)) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    tmp.replace(ARCHIVE_PATH)
assert sha256_file(ARCHIVE_PATH) == ARCHIVE_SHA256, "Checksum архива не совпадает"

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    member = next(m for m in archive.getmembers() if m.name.endswith("/data/ru-RU.jsonl"))
    rows = [json.loads(l) for l in archive.extractfile(member).read().decode("utf-8").splitlines()]
raw = pd.DataFrame(rows)[["id", "partition", "utt", "scenario"]].rename(columns={"utt": "text", "scenario": "label"})

def duplicate_key(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())

raw["key"] = raw["text"].map(duplicate_key)
splits, seen = {}, set()
for part in ("train", "dev", "test"):
    frame = raw[raw["partition"].eq(part)].sort_values("id").drop_duplicates("key")
    frame = frame[~frame["key"].isin(seen)]
    seen.update(frame["key"])
    if part == "train":
        frame = pd.concat([g.sample(n=min(len(g), 300), random_state=SEED)
                           for _, g in frame.groupby("label")])
    splits[part] = frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
train_df, dev_df, test_df = splits["train"], splits["dev"], splits["test"]
LABELS = sorted(train_df["label"].unique())
y_train, y_dev, y_test = (f["label"].to_numpy() for f in (train_df, dev_df, test_df))
print({k: len(v) for k, v in splits.items()}, "| классов:", len(LABELS))
display(pd.DataFrame({p: f["label"].value_counts() for p, f in splits.items()}).loc[LABELS])

Прежде чем хвалить модель, нужна точка отсчёта: классификатор, который всегда отвечает самым частым
классом. Метрика — macro-F1: она усредняет качество по классам, поэтому редкий сценарий весит столько же,
сколько частый, и «угадывать только популярное» не выйдет.

In [ ]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)

dummy = DummyClassifier(strategy="most_frequent").fit(train_df["text"], y_train)
print("Базовая линия (самый частый класс), dev macro-F1:", round(macro_f1(y_dev, dummy.predict(dev_df["text"])), 4))

## Задание 1. Нормализация и стемминг (1,5 балла)
Для TF-IDF «будильник», «будильники» и «будильника» — три разных слова, и редкие формы почти не
получают статистики. Нормализация и стемминг склеивают формы в одну. Проверим, насколько сжимается словарь.

Реализуйте две функции:

* `normalize(text)` — нижний регистр, `ё → е`, удаление всего, кроме букв, цифр и пробелов,
  схлопывание пробелов;
* `stem_text(text)` — `normalize`, затем `wordpunct_tokenize` и `SnowballStemmer("russian")`
  для каждого токена; вернуть строку из основ через пробел.

Следующая ячейка посчитает размер словаря (число уникальных токенов) на train для трёх вариантов:
исходный текст после `wordpunct_tokenize`, `normalize`, `stem_text`.

In [ ]:
stemmer = SnowballStemmer("russian")

def normalize(text: str) -> str:
    text = ...  # TODO: нижний регистр и замена "ё" на "е" (методы .lower() и .replace())
    text = re.sub(r"[^\w\s]|_", " ", text)   # всё, кроме букв, цифр и пробелов, → пробел
    return ...  # TODO: схлопнуть пробелы: разбить по пробелам и склеить через " "

def stem_text(text: str) -> str:
    tokens = wordpunct_tokenize(normalize(text))
    return ...  # TODO: применить stemmer.stem к каждому токену и склеить через " "

assert normalize("  Ёлка,   ПОСТАВЬ будильник!! ") == "елка поставь будильник"
assert stem_text("Поставь будильники") == "постав будильник"

Теперь посчитаем эффект в цифрах: размер словаря до и после обработки.

In [ ]:
vocab_sizes = {
    "raw": len({t for s in train_df["text"] for t in wordpunct_tokenize(s)}),
    "normalize": len({t for s in train_df["text"] for t in normalize(s).split()}),
    "stem_text": len({t for s in train_df["text"] for t in stem_text(s).split()}),
}
display(pd.Series(vocab_sizes, name="vocab_size"))

**Вопрос 1.** Во сколько раз стемминг сократил словарь по сравнению с исходным текстом?

**Ответ:** …

## Задание 2. TF-IDF: слова и символы (2 балла)
TF-IDF — рабочая лошадка: обучается за секунды, не требует GPU, а веса можно прочитать глазами. Сравним
три способа нарезать текст на признаки. Символьные n-граммы устойчивы к опечаткам и окончаниям без всякого
стеммера. `C` — сила регуляризации. Подбираем её на dev, а test не трогаем, чтобы финальная оценка была честной.

Реализуйте `make_tfidf_pipeline(kind, C)`, возвращающую `Pipeline` из `TfidfVectorizer` и
`LogisticRegression(max_iter=2000, C=C, random_state=SEED)`:

* `kind="word"` — `preprocessor=normalize`, словесные n-граммы `(1, 2)`, `min_df=2`, `sublinear_tf=True`;
* `kind="stem"` — то же, но `preprocessor=stem_text`;
* `kind="char"` — `analyzer="char_wb"`, `ngram_range=(2, 5)`, `preprocessor=normalize`, `min_df=2`, `sublinear_tf=True`.

Подберите `C` из `[0.3, 1, 3, 10]` по macro-F1 **на dev** для каждого варианта.
Test в этом задании не используется.

In [ ]:
C_GRID = [0.3, 1, 3, 10]

def make_tfidf_pipeline(kind: str, C: float) -> Pipeline:
    if kind == "char":
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), preprocessor=normalize,
                              min_df=2, sublinear_tf=True)
    else:
        vec = TfidfVectorizer(preprocessor=...,  # TODO: normalize для "word", stem_text для "stem"
                              ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                              token_pattern=r"(?u)\b\w+\b")
    return Pipeline([("tfidf", vec),
                     ("clf", LogisticRegression(max_iter=2000, C=C, random_state=SEED))])

tfidf_rows, tfidf_best = [], {}
for kind in ["word", "stem", "char"]:
    for C in C_GRID:
        t0 = perf_counter()
        pipe = ...   # TODO: make_tfidf_pipeline(kind, C), обученный .fit на train_df["text"] и y_train
        score = ...  # TODO: macro_f1 на dev: истинные y_dev, предсказания pipe.predict(dev_df["text"])
        tfidf_rows.append({"kind": kind, "C": C, "dev_macro_f1": round(score, 4),
                           "n_features": len(pipe.named_steps["tfidf"].vocabulary_),
                           "seconds": round(perf_counter() - t0, 1)})
        if score > tfidf_best.get(kind, (None, -1))[1]:
            tfidf_best[kind] = (pipe, score, C)
tfidf_table = pd.DataFrame(tfidf_rows)
display(tfidf_table.pivot(index="C", columns="kind", values="dev_macro_f1"))
print({k: (round(s, 4), C) for k, (_, s, C) in tfidf_best.items()})

## Задание 3. Векторы предложений (2 балла)
Векторы предложений «знают», что «разбуди меня в семь» и «поставь будильник» — об одном, хотя общих
слов у фраз нет. TF-IDF так не умеет. Кодирование — самая медленная часть ноутбука, поэтому результат
кешируется на диск: повторный запуск будет мгновенным.

Сравните две модели из Sentence Transformers:

* `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` — как на семинаре;
* `intfloat/multilingual-e5-small` — модель E5 ожидает префикс `"query: "` перед каждым текстом.

Реализуйте `encode(model_name, texts)` с нормализацией векторов и кешированием на диск
(`np.save` в `CACHE_DIR`; ключ кеша должен зависеть от модели и списка текстов).
Логистическая регрессия на векторах train и подбор `C` из `C_GRID` по dev — в следующей ячейке, готовым кодом.

In [ ]:
from sentence_transformers import SentenceTransformer

ENCODERS = {
    "minilm": ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ""),
    "e5": ("intfloat/multilingual-e5-small", "query: "),
}
_models = {}

def encode(name: str, texts) -> np.ndarray:
    """Возвращает матрицу (len(texts), dim) нормализованных векторов; кеширует результат на диск."""
    model_id, prefix = ENCODERS[name]
    key = hashlib.sha256(json.dumps([model_id, prefix, list(texts)], ensure_ascii=False).encode()).hexdigest()[:16]
    path = CACHE_DIR / f"hw1_{name}_{key}.npy"
    if path.exists():
        return np.load(path)
    if name not in _models:
        _models[name] = ...  # TODO: SentenceTransformer(model_id, device=DEVICE)
    vectors = ...  # TODO: _models[name].encode(тексты с префиксом prefix + t, batch_size=128,
                   #       normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    ...  # TODO: сохранить vectors в кеш: np.save(path, vectors)
    return vectors

emb = {name: {part: encode(name, frame["text"].tolist()) for part, frame in splits.items()}
       for name in ENCODERS}
print({name: emb[name]["train"].shape for name in emb})

Поверх векторов — та же логистическая регрессия, что и для TF-IDF. Меняем только признаки, чтобы
сравнение было честным.

In [ ]:
emb_rows, emb_best = [], {}
for name in ENCODERS:
    for C in C_GRID:
        clf = LogisticRegression(max_iter=3000, C=C, random_state=SEED).fit(emb[name]["train"], y_train)
        score = macro_f1(y_dev, clf.predict(emb[name]["dev"]))
        emb_rows.append({"encoder": name, "C": C, "dev_macro_f1": round(score, 4)})
        if score > emb_best.get(name, (None, -1))[1]:
            emb_best[name] = (clf, score, C)
display(pd.DataFrame(emb_rows).pivot(index="C", columns="encoder", values="dev_macro_f1"))
print({k: (round(s, 4), C) for k, (_, s, C) in emb_best.items()})

## Задание 4. k ближайших соседей (1 балл)
kNN — самый прозрачный классификатор: «найди похожие размеченные запросы и посмотри, что у них».
Ровно так устроен поиск в RAG, к которому мы придём в блоке 5. Бонус: новый сценарий добавляется
без переобучения — достаточно добавить примеры.

Реализуйте `knn_predict(train_vecs, train_labels, query_vecs, k)` **без sklearn**:
векторы уже нормализованы, поэтому косинусная близость — это скалярное произведение.
Для каждого запроса возьмите `k` ближайших примеров train и выберите метку голосованием;
при равенстве голосов побеждает метка с большей суммой близостей.

Готовый код сравнит `k ∈ {1, 5, 15}` для лучшего энкодера из задания 3 на dev.

In [ ]:
def knn_predict(train_vecs, train_labels, query_vecs, k=5, batch_size=1024):
    train_labels = np.asarray(train_labels)
    predictions = []
    for start in range(0, len(query_vecs), batch_size):
        sims = ...  # TODO: близости батча запросов query_vecs[start:start + batch_size] ко всем train_vecs (матричное умножение @ и .T)
        top = ...   # TODO: индексы k ближайших: np.argpartition(-sims, kth=min(k, sims.shape[1] - 1), axis=1)[:, :k]
        for row, idx in zip(sims, top):
            votes, weight = Counter(), Counter()
            for i in idx:
                votes[train_labels[i]] += 1
                weight[train_labels[i]] += ...  # TODO: близость к соседу i — row[i]
            predictions.append(...)  # TODO: метка с наибольшим числом голосов, при равенстве — с большей суммой
                                     #       близостей: max(votes, key=lambda lab: (votes[lab], weight[lab]))
    return np.array(predictions)

toy_train = np.array([[1, 0], [0.9, 0.1], [0, 1]], dtype=float)
toy_train /= np.linalg.norm(toy_train, axis=1, keepdims=True)
assert list(knn_predict(toy_train, np.array(["a", "a", "b"]), np.array([[1.0, 0.0], [0.0, 1.0]]), k=1)) == ["a", "b"]

Сравним разное число соседей: при `k=1` решение зависит от одного примера и чувствительно к ошибкам
разметки, при большом `k` соседи «размываются» чужими классами.

In [ ]:
best_encoder = max(emb_best, key=lambda n: emb_best[n][1])
knn_rows = []
for k in (1, 5, 15):
    pred = knn_predict(emb[best_encoder]["train"], y_train, emb[best_encoder]["dev"], k=k)
    knn_rows.append({"encoder": best_encoder, "k": k, "dev_macro_f1": round(macro_f1(y_dev, pred), 4)})
display(pd.DataFrame(knn_rows))

## Задание 5. Оценка на test и confusion matrix (1,5 балла)
Test открываем один раз — в самом конце. Если выбирать модель по test, мы подгоняем её под конкретные
примеры, и оценка становится оптимистичной. Самые частые ошибки покажут, где модель ошибается системно.

Выберите **по dev** лучший вариант TF-IDF (лучший вариант на векторах — логистическая регрессия
или kNN — выбирается готовым кодом). Только теперь посчитайте предсказания на test; accuracy и macro-F1
для обеих систем и базовой линии считаются готовым кодом.

Реализуйте `top_confusions(y_true, y_pred, n)` — `n` самых частых пар «истинный → предсказанный»
среди ошибок (DataFrame со столбцами `true`, `pred`, `count`).

In [ ]:
def top_confusions(y_true, y_pred, n=10) -> pd.DataFrame:
    pairs = ...  # TODO: Counter пар (t, p) из zip(y_true, y_pred) только там, где t != p
    return pd.DataFrame([{"true": t, "pred": p, "count": c} for (t, p), c in pairs.most_common(n)])

# Лучшие варианты выбираются только по dev
best_tfidf_kind = ...  # TODO: вариант TF-IDF с лучшим macro-F1 на dev: max(tfidf_best, key=lambda k: tfidf_best[k][1])
best_knn = max(knn_rows, key=lambda r: r["dev_macro_f1"])
use_knn = best_knn["dev_macro_f1"] > emb_best[best_encoder][1]
test_predictions = {
    "dummy": dummy.predict(test_df["text"]),
    "tfidf": ...,  # TODO: предсказания лучшего TF-IDF на test: tfidf_best[best_tfidf_kind][0].predict(test_df["text"])
    "embeddings": (knn_predict(emb[best_encoder]["train"], y_train, emb[best_encoder]["test"], k=best_knn["k"])
                   if use_knn else emb_best[best_encoder][0].predict(emb[best_encoder]["test"])),
}
descr = {"dummy": "most_frequent", "tfidf": f"{best_tfidf_kind}, C={tfidf_best[best_tfidf_kind][2]}",
         "embeddings": f"{best_encoder} + " + (f"kNN k={best_knn['k']}" if use_knn else f"LogReg C={emb_best[best_encoder][2]}")}
final_rows = [{"system": name, "config": descr[name], "test_accuracy": round(accuracy_score(y_test, pred), 4),
               "test_macro_f1": round(macro_f1(y_test, pred), 4)} for name, pred in test_predictions.items()]
display(pd.DataFrame(final_rows))
display(top_confusions(y_test, test_predictions["embeddings"], n=8))

Confusion matrix показывает то же нагляднее: диагональ — доля верно угаданных примеров каждого класса,
яркие клетки вне диагонали — классы, которые модель систематически путает.

In [ ]:
# Confusion matrix лучшей системы, нормированная по строкам (готовый код)
best_name = max([r for r in final_rows if r["system"] != "dummy"], key=lambda r: r["test_macro_f1"])["system"]
cm = confusion_matrix(y_test, test_predictions[best_name], labels=LABELS, normalize="true")
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Purples", vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS)), LABELS, rotation=90); ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("предсказано"); ax.set_ylabel("истинно"); ax.set_title(f"Recall по классам: {best_name}")
fig.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

## Задание 6. Кривая обучения (1 балл)
Разметка стоит денег и времени. Кривая обучения отвечает на практический вопрос: сколько примеров
нужно, чтобы выйти на приемлемое качество, и какой подход выгоднее, когда данных мало.

Обучите лучший TF-IDF вариант и логистическую регрессию на лучшем энкодере на 5%, 10%, 25%, 50%
и 100% train (стратифицированная подвыборка с `random_state=SEED`, гиперпараметры — выбранные
выше). График macro-F1 на **dev** от числа обучающих примеров (логарифмическая ось X) строится готовым кодом.

In [ ]:
from sklearn.model_selection import train_test_split
FRACTIONS = [0.05, 0.1, 0.25, 0.5, 1.0]
curve_rows = []
for frac in FRACTIONS:
    idx = np.arange(len(train_df))
    if frac < 1:
        idx, _ = ...  # TODO: train_test_split(idx, train_size=frac, stratify=y_train, random_state=SEED)
    tf = ...  # TODO: make_tfidf_pipeline(best_tfidf_kind, tfidf_best[best_tfidf_kind][2]),
              #       .fit на train_df["text"].iloc[idx] и y_train[idx]
    lr = ...  # TODO: LogisticRegression(max_iter=3000, C=emb_best[best_encoder][2], random_state=SEED),
              #       .fit на emb[best_encoder]["train"][idx] и y_train[idx]
    curve_rows += [{"n_train": len(idx), "system": "tfidf", "dev_macro_f1": macro_f1(y_dev, tf.predict(dev_df["text"]))},
                   {"n_train": len(idx), "system": best_encoder, "dev_macro_f1": macro_f1(y_dev, lr.predict(emb[best_encoder]["dev"]))}]
curve = pd.DataFrame(curve_rows)
ax = curve.pivot(index="n_train", columns="system", values="dev_macro_f1").plot(marker="o", logx=True, figsize=(7, 4))
ax.set_ylabel("dev macro-F1"); ax.set_title("Кривая обучения"); ax.grid(alpha=0.3); plt.show()
display(curve.pivot(index="n_train", columns="system", values="dev_macro_f1").round(4))

**Вопрос 2.** При каком объёме train векторы предложений сильнее всего выигрывают у TF-IDF?

**Ответ:** …

## Задание 7. Анализ ошибок (1 балл)
Метрики говорят «сколько», ошибки — «почему». Часто выясняется, что модель по-своему права, а спорна
сама разметка: такие случаи никакой моделью не исправить.

Выведите 12 ошибок лучшей системы на test: текст, истинный и предсказанный сценарий,
предсказание второй системы.

In [ ]:
other_name = ...  # TODO: вторая система: "tfidf", если best_name == "embeddings", иначе "embeddings"
errors = ...  # TODO: test_df.assign(pred=предсказания test_predictions[best_name], other=test_predictions[other_name])
errors = ...  # TODO: оставить строки, где pred != label (errors["pred"].ne(errors["label"])),
              #       и столбцы ["text", "label", "pred", "other"] — через errors.loc[условие, столбцы]
print(f"Ошибок {best_name}: {len(errors)} из {len(test_df)}")
display(errors.sample(12, random_state=SEED))

**Вопрос 3.** Какая пара сценариев путается чаще всего и почему, судя по примерам из таблицы?

**Ответ:** …

**Вопрос 4.** Какую систему вы бы выбрали для продакшена — TF-IDF или векторы? Одна причина с числом из ноутбука.

**Ответ:** …